In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.config('spark.driver.host','localhost').appName('06').getOrCreate()

In [13]:
ab = spark.read.csv('C:\\datas\\3rd_week_data\\air_bnb.csv',
                   header = True,
                   inferSchema=True)

In [14]:
ab.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- host_id: string (nullable = true)
 |-- host_name: string (nullable = true)
 |-- neighbourhood_group: string (nullable = true)
 |-- neighbourhood: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- price: string (nullable = true)
 |-- minimum_nights: string (nullable = true)
 |-- number_of_reviews: string (nullable = true)
 |-- last_review: string (nullable = true)
 |-- reviews_per_month: string (nullable = true)
 |-- calculated_host_listings_count: string (nullable = true)
 |-- availability_365: integer (nullable = true)



In [16]:
from pyspark.sql.types import *
ab_df = ab.withColumn('price',ab.price.cast(IntegerType())).withColumn('minimum_nights',ab.minimum_nights.cast(IntegerType()))\
.withColumn('number_of_reviews',ab.number_of_reviews.cast(IntegerType())).withColumn('reviews_per_month',ab.reviews_per_month.cast(IntegerType()))

ab_df.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- host_id: string (nullable = true)
 |-- host_name: string (nullable = true)
 |-- neighbourhood_group: string (nullable = true)
 |-- neighbourhood: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- minimum_nights: integer (nullable = true)
 |-- number_of_reviews: integer (nullable = true)
 |-- last_review: string (nullable = true)
 |-- reviews_per_month: integer (nullable = true)
 |-- calculated_host_listings_count: string (nullable = true)
 |-- availability_365: integer (nullable = true)



In [17]:
from pyspark.sql.functions import *

In [24]:
# neighbourhood_group 열을 grouping하고 count로 집계하기
ab_df.groupby('neighbourhood_group').count().show(10)

+-------------------+-----+
|neighbourhood_group|count|
+-------------------+-----+
|         Douglaston|    1|
|             Queens| 5521|
|              Nadia|    1|
|            Midtown|    4|
|     Hell's Kitchen|    6|
|  Greenwich Village|    2|
|       Clinton Hill|    1|
|   Ditmars Steinway|    2|
|           Longwood|    2|
|        Little Neck|    1|
+-------------------+-----+
only showing top 10 rows



In [25]:
# neighbourhood_group 열을 grouping하고 price의 평균값 집계하기
ab_df.groupby('neighbourhood_group').mean('price').show(10)

+-------------------+------------------+
|neighbourhood_group|        avg(price)|
+-------------------+------------------+
|         Douglaston|               1.0|
|             Queens| 99.60460061583046|
|              Nadia|              NULL|
|            Midtown|               9.0|
|     Hell's Kitchen|1.3333333333333333|
|  Greenwich Village|              55.5|
|       Clinton Hill|              14.0|
|   Ditmars Steinway|               3.5|
|           Longwood|               5.0|
|        Little Neck|               1.0|
+-------------------+------------------+
only showing top 10 rows



In [27]:
# agg() 는 집계함수로 원하는 열과 집계함수를 지정해서 사용하며, 여러개도 사용 가능하다
ab_df.groupBy('neighbourhood_group').agg({'price':'mean','number_of_reviews':'max'}).show(10)

+-------------------+------------------+----------------------+
|neighbourhood_group|        avg(price)|max(number_of_reviews)|
+-------------------+------------------+----------------------+
|         Douglaston|               1.0|                  NULL|
|             Queens| 99.60460061583046|                   629|
|              Nadia|              NULL|                    30|
|            Midtown|               9.0|                  NULL|
|     Hell's Kitchen|1.3333333333333333|                  NULL|
|  Greenwich Village|              55.5|                  NULL|
|       Clinton Hill|              14.0|                  NULL|
|   Ditmars Steinway|               3.5|                  NULL|
|           Longwood|               5.0|                  NULL|
|        Little Neck|               1.0|                  NULL|
+-------------------+------------------+----------------------+
only showing top 10 rows



In [29]:
ab_df.groupBy('neighbourhood_group').agg(mean(ab_df.price).alias('mean_price'),max(ab_df.number_of_reviews).alias('max_reviews')).show(10)

+-------------------+------------------+-----------+
|neighbourhood_group|        mean_price|max_reviews|
+-------------------+------------------+-----------+
|         Douglaston|               1.0|       NULL|
|             Queens| 99.60460061583046|        629|
|              Nadia|              NULL|         30|
|            Midtown|               9.0|       NULL|
|     Hell's Kitchen|1.3333333333333333|       NULL|
|  Greenwich Village|              55.5|       NULL|
|       Clinton Hill|              14.0|       NULL|
|   Ditmars Steinway|               3.5|       NULL|
|           Longwood|               5.0|       NULL|
|        Little Neck|               1.0|       NULL|
+-------------------+------------------+-----------+
only showing top 10 rows



In [32]:
# summary() 데이터 프레임의 통계를 확인하는 명령어
ab_df.summary().select('summary','price','number_of_reviews').show()

+-------+------------------+-----------------+
|summary|             price|number_of_reviews|
+-------+------------------+-----------------+
|  count|             48312|            48176|
|   mean| 152.3312427554231|23.26349219528396|
| stddev|239.34030523202495|44.59808062733653|
|    min|               -74|                0|
|    25%|                69|                1|
|    50%|               105|                5|
|    75%|               175|               23|
|    max|             10000|              629|
+-------+------------------+-----------------+



In [ ]:
sales = spark.read.csv()

In [ ]:
sales.printSchema()

In [ ]:
# pivot table로 원하는 데이터 집계
# Ewallet과 Credit card의 최소 사용횟수, 최대 사용횟수를 도시별로 각각 구함
sales.groupBy('city').pivot('payment',['Ewallet','Credit card']).agg(min(sales.Total).alias('min_payment'),max(sales.Total).alias('max_payment')).show()